# Train Transformer Model with Self-Supervised Learning

This notebook demonstrates how to train the transformer model using self-supervised learning with multiple masking strategies.

## Setup and Imports

In [ ]:
import sys
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Import project modules
from src.config import get_config
from src.data_processing import (
    normalize_channels,
    bandpass_filter,
    split_data_stratified,
    generate_synthetic_egm_data,
    generate_synthetic_labels
)
from src.models.transformer import (
    create_egm_transformer_model,
    PretrainingDatasetSpec,
    LossWeights,
    get_transformer_training_config
)
from src.utils import ensure_output_directory, save_model_config

import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Libraries imported successfully")

## 1. Load Configuration

In [ ]:
config = get_config()
config.load_from_yaml('config.yaml')

print("Configuration loaded:")
print(f"  Data directory: {config.paths['data_dir']}")
print(f"  Transformer epochs: {config.get_model_param('transformer', 'epochs', 50)}")
print("  Training ready to begin")

## 2. Generate or Load Training Data

In [ ]:
# Generate synthetic data for demonstration
print("Generating synthetic training data...")
n_samples = 200
n_channels = config.get_data_param('egm_n_channels', 3)
n_timesteps = config.get_data_param('egm_n_samples', 2500)

X = generate_synthetic_egm_data(
    n_samples=n_samples,
    n_channels=n_channels,
    n_timesteps=n_timesteps,
    random_state=42
)
y = generate_synthetic_labels(
    n_samples=n_samples,
    n_classes=3,
    class_balance=0.30,
    random_state=42
)

print("✓ Generated data:")
print(f"  Signals: {X.shape}")
print(f"  Labels: {y.shape}")

## 3. Preprocess Data (Bandpass + Peak Scaling to 1)

In [ ]:
# Apply preprocessing pipeline
print("Applying preprocessing...")

# Bandpass filter (applied across the full batch)
X_filtered = bandpass_filter(X)
print("✓ Bandpass filtered")

# Scale each channel to unit peak magnitude (original workflow behavior)
X_scaled = normalize_channels(X_filtered, method="peak")
print("✓ Scaled channels to peak absolute value = 1")

# Verify preprocessing
print()
print("Preprocessed data statistics:")
print(f"  Shape: {X_scaled.shape}")
print(f"  Mean: {X_scaled.mean():.6f}")
print(f"  Std: {X_scaled.std():.6f}")
print(f"  Min: {X_scaled.min():.6f}")
print(f"  Max: {X_scaled.max():.6f}")

## 4. Split Data

In [ ]:
# Stratified split for train/val/test
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data_stratified(
    X_scaled,
    y,
    test_ratio=0.1,
    val_ratio=0.1,
    random_state=42
)

print("Data split:")
print(f"  Train: {X_train.shape}")
print(f"  Val: {X_val.shape}")
print(f"  Test: {X_test.shape}")

## 5. Define Transformer Architecture

In [ ]:
# Get model architecture specification
model_spec = create_egm_transformer_model(
    n_channels=n_channels,
    n_timesteps=n_timesteps,
    patch_size=config.get_model_param('transformer', 'patch_size', 50),
    d_model=config.get_model_param('transformer', 'd_model', 256),
    nhead=config.get_model_param('transformer', 'nhead', 8),
    num_encoder_layers=config.get_model_param('transformer', 'num_encoder_layers', 8),
    dim_feedforward=config.get_model_param('transformer', 'dim_feedforward', 1024),
    dropout=config.get_model_param('transformer', 'dropout', 0.1),
    output_dim=config.get_data_param('n_label_classes', 3),
)

print("Transformer Model Architecture:")
print(json.dumps(model_spec, indent=2, default=str)[:800] + "...")

## 6. Define Masking Strategies

In [ ]:
# Show self-supervised masking strategies from model spec
masking_spec = PretrainingDatasetSpec.get_masking_strategies()
augmentation_spec = PretrainingDatasetSpec.get_augmentation_strategies()

print("Masking Strategies:")
for name, spec in masking_spec.items():
    print(f"  {name}: probability={spec['probability']:.0%} | {spec['description']}")

print()
print("Data Augmentation:")
for name, spec in augmentation_spec.items():
    print(f"  {name}: {spec}")

## 7. Define Loss Weights

In [ ]:
# Define multi-task loss components
loss_weights = LossWeights.get_all_weights()

print("Loss Components:")
for name, value in loss_weights.items():
    print(f"  {name}: {value}")

## 8. Training Configuration

In [ ]:
# Get training configuration
train_config = get_transformer_training_config()

print("Training Configuration:")
print(f"  Batch size: {train_config['batch_size']}")
print(f"  Epochs: {train_config['epochs']}")
print(f"  Max learning rate: {train_config['max_learning_rate']}")
print(f"  Optimizer: {train_config['optimizer']}")
print(f"  Warmup steps: {train_config['warmup_steps']}")
print(f"  Scheduler: {train_config['scheduler']}")

## 9. Pre-training Phase

**Note:** This is a pseudo-code template. To train actual models, use TensorFlow/PyTorch.

In [ ]:
# Template for TensorFlow training
template_code = """
# TensorFlow Training Template
import tensorflow as tf
from tensorflow import keras

# Build model (framework-specific implementation)
# model = build_transformer_model(model_spec, masking_spec, loss_weights)

# Compile with multi-task loss
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-4),
#     loss=custom_transformer_loss(loss_weights),
#     metrics=['mae', 'mse']
# )

# Train with masking strategies
# history = model.fit(
#     X_train,
#     X_train,  # Reconstruction target (self-supervised)
#     batch_size=32,
#     epochs=50,
#     validation_data=(X_val, X_val),
#     callbacks=[
#         keras.callbacks.EarlyStopping(monitor='val_loss', patience=10),
#         keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
#     ]
# )

""" 

print("TensorFlow Training Template:")
print(template_code)
print()
print("Status: Template provided. Actual implementation depends on your framework choice.")

## 10. Fine-tuning Phase

In [ ]:
# After pre-training, fine-tune classification heads
finetune_config = """
# Fine-tuning Configuration
FREEZE_ENCODER = True          # Keep encoder weights from pretraining
UNFREEZE_LAST_N_LAYERS = 2    # Fine-tune last 2 encoder layers
FINETUNE_LR = 1e-5            # Lower learning rate than pretraining
SUPERVISION_LOSS_WEIGHT = 1.0 # Weight of classification loss

# Loss: Combined reconstruction + classification
COMBINED_LOSS = {
    'pretraining_loss': 0.5,    # Keep reconstruction component
    'classification_loss': 0.5   # Add supervised learning
}

EPOCHS = 30                     # Fewer epochs for fine-tuning
BATCH_SIZE = 32
"""

print(finetune_config)

## 11. Training Progress Monitoring

In [ ]:
# Metrics to monitor during training
metrics_template = {
    'phase': 'pretraining',
    'epoch': 0,
    'loss': {
        'total': 0.0,
        'cross_channel_recon': 0.0,
        'intra_channel_recon': 0.0,
        'autoregressive_recon': 0.0,
        'contrastive': 0.0,
        'regularization': 0.0
    },
    'validation_loss': 0.0,
    'learning_rate': 1e-4,
    'samples_seen': 0
}

print("Metrics Template for Training Monitoring:")
print(json.dumps(metrics_template, indent=2))

## 12. Model Checkpoint Management

In [ ]:
# Save best model configuration
output_dir = ensure_output_directory("models")
output_path = str(Path(output_dir) / "transformer_config.json")

model_config = {
    'type': 'transformer',
    'architecture': model_spec,
    'masking_strategies': masking_spec,
    'augmentation_strategies': augmentation_spec,
    'loss_weights': loss_weights,
    'training_config': train_config,
    'input_shape': (n_channels, n_timesteps),
    'training_data': {
        'n_train_samples': len(X_train),
        'n_val_samples': len(X_val),
        'n_test_samples': len(X_test),
    },
}

save_model_config(model_config, output_path)
print(f"✓ Model configuration saved to {output_path}")

## 13. Summary: Training Workflow

In [ ]:
print("\n" + "="*60)
print("TRANSFORMER TRAINING WORKFLOW SUMMARY")
print("="*60)
print()
print("SELF-SUPERVISED PRETRAINING SETUP")
print("-" * 60)
print(f"  Data size: {len(X_train)} train samples")
print(f"  Input shape: {(n_channels, n_timesteps)}")
print("  Masking strategies: cross-channel, intra-channel, autoregressive")
print(f"  Epochs: {train_config['epochs']}")
print(f"  Batch size: {train_config['batch_size']}")
print()
print("EVALUATION")
print("-" * 60)
print(f"  Validation set: {len(X_val)} samples")
print(f"  Test set: {len(X_test)} samples")
print("  Metrics: AUC-ROC, F1-score, class-wise sensitivity/specificity")
print()
print("Ready to train. See 05_evaluate_models.ipynb for evaluation.")